# 03 — Supervised learning (binary and multi-class)

**Purpose.** Train candidate classifiers, compare them on the **validation** split, pick operating thresholds and
explain global behaviour. The test split stays untouched until notebook 06.

Candidates (SRS 1.4 / Technical Requirements 4.3): logistic regression, random forest, XGBoost and a linear SVM
(wrapped in calibration so it exposes probabilities). Linear models train on a stratified subsample (`CS_LINEAR_SAMPLE`).
Class imbalance is handled with balanced class/sample weights computed on the training split.

| | |
|---|---|
| **Inputs** | processed splits + fitted preprocessor from notebook 02 |
| **Outputs** | staged models `model_artifacts/staging/<family>/supervised/`, `selection.json`, experiment record |

In [ ]:
# --- Project bootstrap: make backend/app/ml and notebooks/nb_utils importable ---------------------
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not ((_root / "backend" / "app" / "ml").is_dir() and (_root / "notebooks").is_dir()):
    if _root.parent == _root:
        raise RuntimeError("Start Jupyter inside the CyberSentinel repository (for example from notebooks/).")
    _root = _root.parent
for _path in (str(_root / "backend"), str(_root / "notebooks")):
    if _path not in sys.path:
        sys.path.insert(0, _path)

from nb_utils import load_config

CFG = load_config()  # all settings come from CS_* environment variables (see nb_utils/config.py)
CFG.describe()

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.frozen import FrozenEstimator
from sklearn.inspection import permutation_importance
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import train_test_split

from app.ml.artifacts.export import transformed_feature_sources
from app.ml.constants import BENIGN_LABEL, MALICIOUS_LABEL
from app.ml.datasets import get_schema_for_family
from app.ml.evaluation import metrics as M
from nb_utils import io, models, plotting
from nb_utils.experiments import record_experiment, save_json
from nb_utils.preprocessing import build_preprocessor, input_feature_specs

warnings.filterwarnings("ignore")
SCHEMA = get_schema_for_family(CFG.family)
STAGE = CFG.staging_dir / "supervised"
STAGE.mkdir(parents=True, exist_ok=True)
FAST = CFG.fast_mode

In [ ]:
splits_ = io.load_splits(CFG.processed_dir)
prep_report = io.load_json(CFG.processed_dir / "preprocessing_report.json")
preprocessor = io.load_object(CFG.processed_dir / "preprocessor.joblib")
FEATURES = [f["name"] for f in prep_report["input_features"]]
SOURCES = transformed_feature_sources(preprocessor)
train, val = splits_["train"], splits_["validation"]
X_train = preprocessor.transform(train[FEATURES])
X_val = preprocessor.transform(val[FEATURES])

MC_CLASSES = models.class_order(train["label_category"])
BIN_CLASSES = [BENIGN_LABEL, MALICIOUS_LABEL]
CLASSES = {"multiclass": MC_CLASSES, "binary": BIN_CLASSES}
y = {
    "multiclass": (models.encode(train["label_category"], MC_CLASSES), models.encode(val["label_category"], MC_CLASSES)),
    "binary": (train["label_binary"].to_numpy(), val["label_binary"].to_numpy()),
}
print("train", X_train.shape, "| validation", X_val.shape)
print("multi-class classes (index order):", MC_CLASSES)

## 1. Evaluation helper (validation only)
ROC-AUC/PR-AUC are reported for the attack-vs-benign score; one-vs-rest ROC-AUC skips classes absent from the split (undefined), and undefined values are reported as missing rather than guessed.

In [ ]:
def evaluate(model, X, y_true, classes):
    proba = model.predict_proba(X)
    labels_true = np.asarray(classes, dtype=object)[y_true]
    labels_pred = np.asarray(classes, dtype=object)[proba.argmax(axis=1)]
    report = M.classification_report_dict(labels_true, labels_pred, classes)
    attack_score = 1.0 - proba[:, classes.index(BENIGN_LABEL)]
    is_attack = (labels_true != BENIGN_LABEL).astype(int)
    result = {
        "macro_precision": report["macro"]["precision"],
        "macro_recall": report["macro"]["recall"],
        "macro_f1": report["macro"]["f1"],
        "weighted_f1": report["weighted"]["f1"],
        "accuracy": report["accuracy"],
        "false_alarm_rate": M.false_alarm_rate(labels_true, labels_pred, BENIGN_LABEL),
        "attack_roc_auc": M.roc_auc_safe(is_attack, attack_score),
        "attack_pr_auc": M.pr_auc_safe(is_attack, attack_score),
    }
    if len(classes) > 2:
        result["ovr_roc_auc_macro"] = M.multiclass_roc_auc_ovr(labels_true, proba, classes)["macro"]
    latency = M.measure_latency(model.predict_proba, X[: min(len(X), 10_000)])
    result["ms_per_record"] = latency["ms_per_record"]
    result["records_per_second"] = latency["records_per_second"]
    return result, report, M.confusion(labels_true, labels_pred, classes)

## 2. Train candidates

In [ ]:
factories = {
    "logistic_regression": lambda task: models.make_logistic_regression(CFG.seed),
    "random_forest": lambda task: models.make_random_forest(CFG.seed, CFG.n_jobs, FAST),
    "xgboost": lambda task: models.make_xgboost(CFG.seed, len(CLASSES[task]), CFG.n_jobs, FAST),
    "linear_svm": lambda task: models.make_linear_svm(CFG.seed),
}
rows, reports, confusions, fitted, train_seconds = [], {}, {}, {}, {}
for task in ("multiclass", "binary"):
    y_tr, y_va = y[task]
    for name, factory in factories.items():
        key = f"{task}__{name}"
        model = factory(task)
        idx = np.arange(len(y_tr))
        fit_kwargs = {}
        if name in ("logistic_regression", "linear_svm"):
            idx = models.stratified_indices(y_tr, CFG.linear_sample, CFG.seed, min_per_class=5)
        if name == "xgboost":
            fit_kwargs["sample_weight"] = models.balanced_weights(y_tr)
            if model.get_params().get("early_stopping_rounds"):
                fit_kwargs.update(eval_set=[(X_val, y_va)], verbose=False)  # early stopping on validation, never test
        try:
            model, seconds = models.fit_timed(model, X_train[idx], y_tr[idx], **fit_kwargs)
        except ValueError as exc:  # e.g. a class too small for the SVM's internal calibration folds
            print(f"{key}: skipped — {exc}")
            continue
        metrics_val, report, cm = evaluate(model, X_val, y_va, CLASSES[task])
        fitted[key], reports[key], confusions[key], train_seconds[key] = model, report, cm, round(seconds, 3)
        rows.append({"task": task, "model": name, "split": "validation", "train_rows": int(len(idx)),
                     "train_seconds": round(seconds, 3), "size_mb": round(models.serialized_size_bytes(model) / 1e6, 3), **metrics_val})
        io.save_object(model, STAGE / f"{key}.joblib")
        print(f"{key:32s} macro-F1={metrics_val['macro_f1']}  false-alarm={metrics_val['false_alarm_rate']}  ({seconds:.1f}s)")
results = pd.DataFrame(rows)
results.sort_values(["task", "macro_f1"], ascending=[True, False])

## 3. Model selection on the validation split
Primary criterion: macro-F1 (treats rare attack classes equally); tie-breaker: lower false-alarm rate.

In [ ]:
selection = {}
for task in ("multiclass", "binary"):
    ranked = results[results["task"] == task].sort_values(["macro_f1", "false_alarm_rate"], ascending=[False, True])
    best = ranked.iloc[0]
    selection[task] = {"model": best["model"], "key": f"{task}__{best['model']}",
                       "validation_macro_f1": best["macro_f1"], "validation_false_alarm_rate": best["false_alarm_rate"]}
    cm = confusions[selection[task]["key"]]
    plotting.confusion_heatmap(cm["matrix"], cm["labels"], f"{selection[task]['key']} (validation, row-normalized)",
                               CFG.figures_dir / f"03_confusion_{task}.png")
selection

## 4. Binary decision threshold (validation)

In [ ]:
binary_model = fitted[selection["binary"]["key"]]
binary_scores = binary_model.predict_proba(X_val)[:, 1]
analysis = M.threshold_analysis(y["binary"][1], binary_scores, thresholds=np.round(np.linspace(0.05, 0.95, 19), 2))
chosen = M.select_threshold(analysis, strategy="max_f1")
selection["binary"].update({"decision_threshold": float(chosen["threshold"]), "threshold_method": "maximum F1 on the validation split", "min_confidence": 0.0})
plotting.threshold_curve(analysis, "Binary threshold analysis (validation)", CFG.figures_dir / "03_threshold_binary.png")
pd.DataFrame(analysis)[["threshold", "precision", "recall", "f1", "false_positive_rate"]]

## 5. Confidence threshold for `needs_review` (multi-class)
Records whose top-class score is below `min_confidence` are returned as `needs_review` instead of a possibly fabricated attack label (Workflow Manual 3.3). The table shows the coverage/accuracy trade-off on validation; adjust `MIN_CONFIDENCE` if needed.

In [ ]:
mc_model = fitted[selection["multiclass"]["key"]]
mc_proba = mc_model.predict_proba(X_val)
confidence, correct = mc_proba.max(axis=1), mc_proba.argmax(axis=1) == y["multiclass"][1]
trade_off = pd.DataFrame([
    {"min_confidence": t, "coverage": float((confidence >= t).mean()),
     "accuracy_on_covered": float(correct[confidence >= t].mean()) if (confidence >= t).any() else None,
     "routed_to_review": int((confidence < t).sum())}
    for t in (0.0, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9)
])
MIN_CONFIDENCE = 0.5
selection["multiclass"]["min_confidence"] = MIN_CONFIDENCE
trade_off

## 6. Calibration diagnostics
Scores are presented as *uncalibrated* unless calibration is performed and evaluated (`CS_CALIBRATE=1`: isotonic calibration fitted on one half of the validation split and evaluated on the other half).

In [ ]:
attack_score_val = 1.0 - mc_proba[:, MC_CLASSES.index(BENIGN_LABEL)]
y_val_bin = val["label_binary"].to_numpy()
frac_pos, mean_pred = calibration_curve(y_val_bin, attack_score_val, n_bins=10, strategy="quantile")
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], "k--", label="perfectly calibrated")
ax.plot(mean_pred, frac_pos, marker="o", label=selection["multiclass"]["model"])
ax.set_xlabel("mean predicted attack score"); ax.set_ylabel("observed attack fraction"); ax.legend(); ax.set_title("Reliability (validation)")
fig.savefig(CFG.figures_dir / "03_reliability.png", dpi=120)
print("Brier score of the uncalibrated attack score (validation):", round(brier_score_loss(y_val_bin, attack_score_val), 5))

CALIBRATION = None
if CFG.calibrate:
    labels_val = y["multiclass"][1]
    stratify = labels_val if np.bincount(labels_val).min(initial=2) >= 2 else None
    cal_idx, hold_idx = train_test_split(np.arange(len(labels_val)), test_size=0.5, stratify=stratify, random_state=CFG.seed)
    calibrated = CalibratedClassifierCV(FrozenEstimator(mc_model), method="isotonic").fit(X_val[cal_idx], labels_val[cal_idx])
    held = 1.0 - calibrated.predict_proba(X_val[hold_idx])[:, MC_CLASSES.index(BENIGN_LABEL)]
    CALIBRATION = {"method": "isotonic (CalibratedClassifierCV over FrozenEstimator)", "evaluated_on": "held-out half of the validation split",
                   "brier_score": float(brier_score_loss(y_val_bin[hold_idx], held))}
    io.save_object(calibrated, STAGE / "multiclass__calibrated.joblib")
    print("Brier score after calibration (held-out validation half):", round(CALIBRATION["brier_score"], 5))
else:
    print("Calibration disabled: the API will keep `probability` empty and label scores as uncalibrated.")
selection["multiclass"]["calibration"] = CALIBRATION

## 7. Global feature importance
Model-reported importance and permutation importance (macro-F1 drop on a validation sample) are aggregated from transformed columns back to input features. Importance describes model behaviour, not causation.

In [ ]:
def by_source(values):
    return pd.Series(values, index=SOURCES).groupby(level=0).sum().sort_values(ascending=False)

builtin = by_source(mc_model.feature_importances_) if hasattr(mc_model, "feature_importances_") else None
sample_idx = models.stratified_indices(y["multiclass"][1], 500 if FAST else 5000, CFG.seed)
perm = permutation_importance(mc_model, X_val[sample_idx], y["multiclass"][1][sample_idx], scoring="f1_macro",
                              n_repeats=3, random_state=CFG.seed, n_jobs=1)
permutation = by_source(perm.importances_mean)
GLOBAL_IMPORTANCE = builtin if builtin is not None else permutation
plotting.feature_importance_bar(permutation, "Permutation importance (validation, macro-F1 drop)", CFG.figures_dir / "03_permutation_importance.png")
pd.DataFrame({"model_reported": builtin, "permutation": permutation}).sort_values("permutation", ascending=False).head(20)

In [ ]:
try:  # optional: SHAP summary for tree models (pip install shap)
    import shap

    sample = X_val[models.stratified_indices(y["multiclass"][1], 200, CFG.seed)]
    values = np.abs(np.asarray(shap.TreeExplainer(mc_model).shap_values(sample)))
    feature_axis = [i for i, n in enumerate(values.shape) if n == sample.shape[1]][-1]
    mean_abs = values.mean(axis=tuple(i for i in range(values.ndim) if i != feature_axis))
    shap_importance = by_source(mean_abs)
    plotting.feature_importance_bar(shap_importance, "Mean |SHAP| (validation sample)", CFG.figures_dir / "03_shap_importance.png")
    display(shap_importance.head(15))
except Exception as exc:  # noqa: BLE001 - SHAP is optional
    print("SHAP summary skipped:", type(exc).__name__, exc)

## 8. Shortcut-feature ablation (validation)
Retrains a small random forest with and without the schema's shortcut candidates (preprocessor refitted on train only) to show how much the model relies on them.

In [ ]:
ABLATION = None
present = [c for c in SCHEMA.shortcut_feature_candidates if c in FEATURES]
if present:
    reduced = [f for f in FEATURES if f not in present]
    reduced_pre = build_preprocessor(input_feature_specs(SCHEMA, reduced, train)).fit(train[reduced])
    with_rf = models.make_random_forest(CFG.seed, CFG.n_jobs, fast=True).fit(X_train, y["multiclass"][0])
    without_rf = models.make_random_forest(CFG.seed, CFG.n_jobs, fast=True).fit(reduced_pre.transform(train[reduced]), y["multiclass"][0])
    with_metrics, *_ = evaluate(with_rf, X_val, y["multiclass"][1], MC_CLASSES)
    without_metrics, *_ = evaluate(without_rf, reduced_pre.transform(val[reduced]), y["multiclass"][1], MC_CLASSES)
    ABLATION = {"removed_features": present, "macro_f1_with": with_metrics["macro_f1"], "macro_f1_without": without_metrics["macro_f1"],
                "false_alarm_with": with_metrics["false_alarm_rate"], "false_alarm_without": without_metrics["false_alarm_rate"]}
ABLATION

## 9. Save the selection and the validation experiment record

In [ ]:
selection["multiclass"].update({
    "classes": MC_CLASSES,
    "global_feature_importance": [{"feature": f, "importance": float(v)} for f, v in GLOBAL_IMPORTANCE.head(30).items()],
})
selection["binary"]["classes"] = BIN_CLASSES
selection["train_seconds"] = train_seconds
selection["validation_reports"] = {selection[t]["key"]: reports[selection[t]["key"]] for t in ("multiclass", "binary")}
selection["shortcut_ablation"] = ABLATION
save_json(selection, STAGE / "selection.json")
record_experiment(
    experiment_id=f"{CFG.family}-supervised-validation",
    experiment_type="supervised_comparison",
    title="Supervised candidates (validation split)",
    family=CFG.family,
    rows=results,
    columns={"macro_f1": "macro-averaged F1", "false_alarm_rate": "share of benign flows predicted as attacks",
             "attack_roc_auc": "ROC-AUC of 1 - P(benign)", "ms_per_record": "median predict_proba latency per record (ms)",
             "size_mb": "compressed joblib size (MB)", "train_seconds": "wall-clock training time"},
    config={"seed": CFG.seed, "linear_sample": CFG.linear_sample, "fast_mode": FAST, "input_features": len(FEATURES)},
    summary={"selected": {t: selection[t]["model"] for t in ("multiclass", "binary")}, "shortcut_ablation": ABLATION},
    notes=["Validation split only; final test metrics are produced in 06_model_evaluation.",
           "Linear models were trained on a stratified subsample of the training split."],
    experiments_dir=CFG.experiments_dir,
    reports_dir=CFG.reports_dir,
)
print("selection saved to", STAGE / "selection.json")